# Testing Evolutionary algorithms with DEAP

**S. Doncieux, Sorbonne University, 2026, [stephane.doncieux@sorbonne-universite.fr](mailto:stephane.doncieux@sorbonne-universite.fr)**

_UE Robotique et Apprentissage, Master AI2D_

## Introduction to DEAP

Mostly extracted from https://deap.readthedocs.io/en/master/overview.html

In [7]:
#!pip3 install deap matplotlib
import sys
print(sys.executable)
!{sys.executable} -m pip install deap matplotlib

/opt/homebrew/Cellar/jupyterlab/4.5.6/libexec/bin/python
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 10.8 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 11.4 MB/s  0:00:00 11.7 MB/s eta 0:00:01
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 11.1 MB/s  0:00:00 11.6 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 12.4 MB/s  0:00:002.9 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 11.3 MB/s  0:00:001.6 MB/s eta 0:00:01
  Created wheel for deap: filename=deap-1.4.3-cp314-cp314-macosx_26_0_arm64.whl size=104380 sha256=11fea098e82e817096aafd24554effa9cdaad16f84a023915b35a7c600442ff8
  Stored in directory: /Users/doncieux/Library/Caches/pip/wheels/85/9f/

In [8]:
%matplotlib inline
import matplotlib.pyplot as plt
import pickle
import numpy as np
import importlib
import random

random.seed()

from deap import base, creator, benchmarks

from deap import tools


## A simple example

In [26]:
## We create the types we need ##

# Fitness that is minimized (-1 weight)
if (hasattr(creator, "FitnessMin")):
    # Deleting any previous definition (to avoid warning message)
    del creator.FitnessMin
creator.create("FitnessMin", base.Fitness, weights=(1.0,))

# Individual that uses this fitness
if (hasattr(creator, "Individual")):
    # Deleting any previous definition (to avoid warning message)
    del creator.Individual
creator.create("Individual", list, fitness=creator.FitnessMin)

## Tool initialization ##
IND_SIZE = 5

# toolbox is a container, each registered function can be called later on. Example:
# toolbox.register("my_function", my_great_function, default_param=42)
# toobox.my_function(...) calls my_great_function(...)
# some parameters with default values can be defined when registering the function, 
# they are then transmitted to it when it is called 
# (in the example, the param default_param is transmitted to the function with the value 42)
toolbox = base.Toolbox()

# parameters are initialized between 0 and 1
toolbox.register("attribute", random.random)

# individuals are made with IND_SIZE parameters
toolbox.register("individual", tools.initRepeat, creator.Individual,
                 toolbox.attribute, n=IND_SIZE)

# the population is a list of individuals
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def evaluate(individual):
    return sum(individual), # this is a simple basic fitness for testing purposes

# 2 points crossover
toolbox.register("mate", tools.cxTwoPoint)

# gaussian mutation
toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=1, indpb=0.1)

# Tournament selection: select the best out of X randomly selected individuals (with X=3)
toolbox.register("select", tools.selTournament, tournsize=3)

# Evaluation function to use
toolbox.register("evaluate", evaluate)


In [27]:
def main(NGEN=40, POPSIZE=10):

    # Initialising the population
    pop = toolbox.population(n=POPSIZE)
    CXPB, MUTPB = 0.5, 0.2

    # Evaluate the entire population
    fitnesses = list(map(toolbox.evaluate, pop))
    for ind, fit in zip(pop, fitnesses):
        ind.fitness.values = fit

    for g in range(NGEN):
        # Select the next generation individuals
        offspring = toolbox.select(pop, len(pop)) 
        # WARNING: makes sense if we use tournament selection, 
        # does not make sense if we use an elitist selection (can you guess why ?)
        
        # Clone the selected individuals
        offspring = list(map(toolbox.clone, offspring))

        # Apply crossover and mutation on the offspring
        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < CXPB:
                toolbox.mate(child1, child2)
                del child1.fitness.values
                del child2.fitness.values

        for mutant in offspring:
            if random.random() < MUTPB:
                toolbox.mutate(mutant)
                del mutant.fitness.values

        # Evaluate the individuals with an invalid fitness
        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        fitnesses = map(toolbox.evaluate, invalid_ind)
        for ind, fit in zip(invalid_ind, fitnesses):
            ind.fitness.values = fit

        # The population is entirely replaced by the offspring
        pop[:] = offspring

    return pop

In [28]:
pop=main(10)

In [29]:
for i,ind in enumerate(pop):
    print("Indiv[%d]=%f"%(i,evaluate(ind)[0])+" "+str(ind))

Indiv[0]=6.215214 [1.1400639679515605, 2.6374507963191496, 0.898567274973651, 0.8135403354641794, 0.7255918592775056]
Indiv[1]=6.814092 [0.9778337965130898, 2.6374507963191496, 0.898567274973651, 1.5746478478818258, 0.7255918592775056]
Indiv[2]=6.814092 [0.9778337965130898, 2.6374507963191496, 0.898567274973651, 1.5746478478818258, 0.7255918592775056]
Indiv[3]=6.814092 [0.9778337965130898, 2.6374507963191496, 0.898567274973651, 1.5746478478818258, 0.7255918592775056]
Indiv[4]=6.814092 [0.9778337965130898, 2.6374507963191496, 0.898567274973651, 1.5746478478818258, 0.7255918592775056]
Indiv[5]=6.215214 [1.1400639679515605, 2.6374507963191496, 0.898567274973651, 0.8135403354641794, 0.7255918592775056]
Indiv[6]=6.814092 [0.9778337965130898, 2.6374507963191496, 0.898567274973651, 1.5746478478818258, 0.7255918592775056]
Indiv[7]=6.814092 [0.9778337965130898, 2.6374507963191496, 0.898567274973651, 1.5746478478818258, 0.7255918592775056]
Indiv[8]=6.814092 [0.9778337965130898, 2.637450796319149